# Клиенты торгового эквайринга на дату

Одна строка — один договор SA, открытый в выбранный день: `d_valid_from <= дата` и дата закрытия пустая или `>= дата`. То же у привязки `cf_ter_type = 'P'`.

Колонки: ИНН, ЦФТ ID, номер договора, дата открытия, дата закрытия, РФ. Пустая дата закрытия — договор на эту дату ещё открыт. РФ приводится к виду `Чувашский РФ`, как `normalize_filial_rf` в `final_df`. Если после этого осталось только `РФ`, в файл пишется `ЦРМБ`. ИНН из 9 цифр дополняется слева до 10, из 11 — до 12.

Лист `контроль` — договоры без ЦФТ ID, пустой филиал в озере и число организаций ЦФТ (`class_id = 'CL_ORG'`) без договора ТЭ на эту дату.

Смените `AS_OF_DATE` и запустите обе ячейки. Файл `trade_acquiring_clients_YYYY-MM-DD.xlsx` ляжет в `/home/jovyan/documents/Equaring`.



In [ ]:
import getpass
import re
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

AS_OF_DATE = '2026-05-31'
OUT_DIR = Path('/home/jovyan/documents/Equaring')

as_of = pd.to_datetime(AS_OF_DATE).strftime('%Y-%m-%d')
REPORT_MONTH = as_of[:7]
print(as_of)

LAKE_USER = input('Логин Impala: ').strip()
LAKE_PASSWORD = getpass.getpass('Пароль Impala: ')

user_params = {'user_name': LAKE_USER, 'password': LAKE_PASSWORD}

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params=user_params,
)
imp._init_connection()
print('imp ready, user =', LAKE_USER)


In [ ]:
def normalize_filial_rf(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\s+', ' ', str(v)).strip()
    m = re.search(r'(?i)рф', s)
    if m:
        s = s[:m.end()]
    s = re.sub(r'(?i)\b(ао|пао|оао|зао)\b.*$', '', s).strip(' ,;.-')
    if not s:
        return None
    s = s[:1].upper() + s[1:].lower()
    s = re.sub(r'(?i)рф', 'РФ', s)
    if s == 'РФ':
        return 'ЦРМБ'
    return s


sql = '''-- Договоры SA, открытые на дату: номер, даты, ИНН. 9 цифр -> 10, 11 -> 12.
WITH sa AS (
  SELECT DISTINCT
    a.n_agr,
    a.c_agr_number,
    a.d_valid_from,
    a.d_valid_to,
    CASE length(regexp_replace(c.c_inn, '[^0-9]', ''))
      WHEN 9 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 10, '0')
      WHEN 11 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 12, '0')
      ELSE regexp_replace(c.c_inn, '[^0-9]', '')
    END AS inn
  FROM ods_alpha.scd1_agreements a
  JOIN ods_alpha.scd1_companies c
    ON c.n_cmp = a.n_cmp_client
  WHERE upper(trim(a.acq_class)) = 'SA'
    AND a.d_valid_from <= '{as_of}'
    AND (a.d_valid_to IS NULL OR a.d_valid_to >= '{as_of}')
    AND coalesce(a.ods_deleted_flg, '0') <> '1'
    AND coalesce(c.ods_deleted_flg, '0') <> '1'
    AND c.c_inn IS NOT NULL
    AND EXISTS (
      SELECT 1
      FROM ods_alpha.scd1_agr_terms t
      WHERE t.n_agr = a.n_agr
        AND t.d_valid_from <= '{as_of}'
        AND (t.d_valid_to IS NULL OR t.d_valid_to >= '{as_of}')
        AND upper(trim(t.cf_ter_type)) = 'P'
        AND coalesce(t.ods_deleted_flg, '0') <> '1'
    )
),
-- Одна живая карточка OCRM на ИНН, самая новая.
ocrm AS (
  SELECT inn, row_id
  FROM (
    SELECT
      s.inn,
      soe.row_id,
      row_number() OVER (
        PARTITION BY s.inn
        ORDER BY soe.created DESC, soe.row_id DESC
      ) AS rn
    FROM (SELECT DISTINCT inn FROM sa) s
    JOIN ocrm_ul.s_org_ext soe
      ON CASE length(regexp_replace(soe.x_inn, '[^0-9]', ''))
           WHEN 9 THEN lpad(regexp_replace(soe.x_inn, '[^0-9]', ''), 10, '0')
           WHEN 11 THEN lpad(regexp_replace(soe.x_inn, '[^0-9]', ''), 12, '0')
           ELSE regexp_replace(soe.x_inn, '[^0-9]', '')
         END = s.inn
    WHERE soe.x_removed_flg = 'N'
      AND soe.x_duplicate_flg = 'N'
  ) t
  WHERE rn = 1
),
-- Идентификатор CDI по карточке OCRM.
cdi AS (
  SELECT inn, cdi_id
  FROM (
    SELECT
      o.inn,
      e.party_id AS cdi_id,
      row_number() OVER (PARTITION BY o.inn ORDER BY e.party_id DESC) AS rn
    FROM ocrm o
    JOIN cdiul.ext_id_org e
      ON e.cmo_ext_party_source_id = o.row_id
     AND e.cmo_ext_source_system LIKE 'OCRM%'
  ) t
  WHERE rn = 1
),
-- ЦФТ ID той же организации в CDI.
cft AS (
  SELECT c.inn, e.cmo_ext_party_source_id AS cft_id
  FROM cdi c
  JOIN cdiul.ext_id_org e
    ON e.party_id = c.cdi_id
   AND e.cmo_ext_source_system LIKE 'CFT%'
),
-- Подпись филиала по ЦФТ: мерчант -> подразделение -> филиал.
filial AS (
  SELECT cft_id, filial_rf
  FROM (
    SELECT
      c.cft_id,
      br.c_shortlabel AS filial_rf,
      row_number() OVER (
        PARTITION BY c.cft_id
        ORDER BY CASE WHEN m.c_depart IS NULL THEN 1 ELSE 0 END, m.id DESC
      ) AS rn
    FROM (SELECT DISTINCT cft_id FROM cft) c
    JOIN ods.scd1_z_r2_ip_merchants m
      ON cast(m.c_cl_org AS string) = c.cft_id
    LEFT JOIN ods.scd1_z_depart dep ON dep.id = m.c_depart
    LEFT JOIN ods.scd1_z_branch br ON br.id = dep.c_filial
    WHERE coalesce(m.ods_deleted_flg, '0') <> '1'
  ) t
  WHERE rn = 1
),
-- Одна строка на договор. Если ЦФТ несколько, берётся тот, где филиал заполнен.
picked AS (
  SELECT
    s.inn,
    c.cft_id,
    s.c_agr_number,
    s.d_valid_from,
    s.d_valid_to,
    f.filial_rf,
    row_number() OVER (
      PARTITION BY s.n_agr
      ORDER BY CASE WHEN f.filial_rf IS NULL THEN 1 ELSE 0 END, c.cft_id
    ) AS rn
  FROM sa s
  LEFT JOIN cft c ON c.inn = s.inn
  LEFT JOIN filial f ON f.cft_id = c.cft_id
)
-- Шесть полей выгрузки.
SELECT
  inn,
  cft_id,
  c_agr_number,
  d_valid_from,
  d_valid_to,
  filial_rf
FROM picked
WHERE rn = 1'''
sql = sql.format(as_of=as_of)

with imp:
    imp.execute('set MEM_LIMIT=16g')
    df = imp.fetch(sql)

df = df.copy()
df.columns = [
    'ИНН',
    'ЦФТ ID',
    'Номер договора',
    'Дата открытия договора',
    'Дата закрытия договора',
    'РФ, в котором обслуживается клиент',
]
df['РФ в озере'] = df['РФ, в котором обслуживается клиент']
df['РФ, в котором обслуживается клиент'] = df['РФ в озере'].apply(normalize_filial_rf)
df['ИНН'] = df['ИНН'].map(lambda v: None if pd.isna(v) else str(v).strip())

def _blank(s):
    return s.isna() | s.astype('string').str.strip().isin(['', 'None', 'nan', '<NA>'])

no_cft = _blank(df['ЦФТ ID'])
no_rf_lake = _blank(df['РФ в озере'])

def _stat(name, mask):
    part = df.loc[mask]
    return {
        'показатель': name,
        'договоров': int(mask.sum()),
        'клиентов ИНН': int(part['ИНН'].nunique(dropna=True)),
        'клиентов ЦФТ': None,
    }

te_ids = [
    str(v).strip().replace("'", '')
    for v in df.loc[~no_cft, 'ЦФТ ID'].dropna().unique()
]
id_sql = ', '.join("'" + v + "'" for v in te_ids) or "''"
sql_without_te = f'''
SELECT count(*) AS clients_without_te
FROM (
  SELECT DISTINCT cast(id AS string) AS cft_id
  FROM ods.scd1_z_client
  WHERE class_id = 'CL_ORG'
    AND c_inn IS NOT NULL
) o
WHERE o.cft_id NOT IN ({id_sql})
'''
with imp:
    imp.execute('set MEM_LIMIT=16g')
    without_te = imp.fetch(sql_without_te)
clients_without_te = int(without_te.iloc[0, 0])

control = pd.DataFrame([
    _stat('Всего', pd.Series(True, index=df.index)),
    _stat('Без ЦФТ ID', no_cft),
    _stat('Пустой РФ в озере', no_rf_lake),
    _stat('Без ЦФТ ID и пустой РФ в озере', no_cft & no_rf_lake),
    {
        'показатель': 'Клиентов ЦФТ без ТЭ',
        'договоров': None,
        'клиентов ИНН': None,
        'клиентов ЦФТ': clients_without_te,
    },
])

out = OUT_DIR / ('trade_acquiring_clients_%s.xlsx' % as_of)
out.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(out, engine='openpyxl') as writer:
    df.drop(columns=['РФ в озере']).to_excel(writer, index=False, sheet_name='договоры')
    control.to_excel(writer, index=False, sheet_name='контроль')
print('строк', len(df))
print(out)
display(control)
df.head(20)


## Импорт Excel

Читает записанный файл. ИНН и ЦФТ ID загружаются как текст, чтобы ведущий ноль не снялся при открытии.


In [ ]:
excel_df = pd.read_excel(
    out,
    sheet_name='договоры',
    dtype={'ИНН': str, 'ЦФТ ID': str},
)
print('файл', out)
print('строк', len(excel_df))
excel_df.head(20)


## Пустой РФ: сырое значение из озера и филиал из отчёта

Берутся строки, где после нормализации филиал пустой. Рядом сырой `c_shortlabel` из озера и филиал из отчёта Excel за тот же месяц. Так видно, обнулила значение нормализация или в озере его уже не было.


In [ ]:
rf = 'РФ, в котором обслуживается клиент'
raw = 'РФ в озере'
DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
excel_by_month = {
    '2026-05': (DATA_DIR / '05_Май_2026.xlsx', 0),
    '2026-06': (DATA_DIR / '06_Июнь_2026.xlsx', 0),
    '2026-07': (DATA_DIR / '07_Июль_2026.xlsx', 0),
}

def norm_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\.0$', '', str(v).strip())
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        return s.zfill(10)
    if len(s) == 11:
        return s.zfill(12)
    return s

def pick(columns, names):
    low = {str(c).strip().lower(): c for c in columns}
    for name in names:
        if name.lower() in low:
            return low[name.lower()]
    raise KeyError(list(columns))

path, header = excel_by_month[REPORT_MONTH]
report = pd.read_excel(path, header=header)
inn_col = pick(report.columns, ['ИНН', 'ИНН клиента', 'inn'])
fil_col = pick(report.columns, ['Филиал', 'Региональный филиал', 'Филиал договора', 'filial_rf'])
con_col = pick(report.columns, ['Номер договора', '№ договора', 'contract_number'])
report = report.copy()
report['ИНН'] = report[inn_col].map(norm_inn)
report['Номер договора'] = report[con_col].map(lambda v: None if pd.isna(v) else str(v).strip())
report['Филиал в отчёте'] = report[fil_col]

empty = df[df[rf].isna()].copy()
empty['ИНН'] = empty['ИНН'].map(norm_inn)
empty['Номер договора'] = empty['Номер договора'].map(lambda v: None if pd.isna(v) else str(v).strip())
empty['сырое в озере'] = empty[raw].map(lambda v: '(пусто в озере)' if pd.isna(v) or str(v).strip() == '' else str(v))

lake_blank = empty[raw].isna() | empty[raw].astype('string').str.strip().isin(['', 'None', 'nan', '<NA>'])
print('пусто после нормализации', len(empty))
print('из них пусто уже в озере', int(lake_blank.sum()))

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
display(empty['сырое в озере'].value_counts(dropna=False).rename_axis('РФ в озере').reset_index(name='договоров'))

samples = (
    empty.sort_values(['сырое в озере', 'ИНН'])
    .groupby('сырое в озере', dropna=False, group_keys=False)
    .head(5)
)
cmp = samples.merge(
    report[['ИНН', 'Номер договора', 'Филиал в отчёте']],
    on=['ИНН', 'Номер договора'],
    how='left',
)
cmp[['ИНН', 'Номер договора', 'сырое в озере', 'Филиал в отчёте']]
